Step 1 — List Available Embedding Models

In [15]:
import json
import boto3
import sagemaker

from sagemaker.jumpstart.model import JumpStartModel

print("SageMaker SDK:", sagemaker.__version__)

SageMaker SDK: 2.257.3


In [12]:
from sagemaker.jumpstart.notebook_utils import list_jumpstart_models

print("JumpStart imported successfully")

JumpStart imported successfully


In [14]:
models = list_jumpstart_models()

for model in models:
    if "embed" in model.lower() or "bge" in model.lower() or "e5" in model.lower():
        print(model)

cohere-embed-english
cohere-embed-light-english
cohere-embed-light-multilingual
cohere-embed-multilingual
cohere-embed-v4-0
huggingface-sentencesimilarity-bge-base-en
huggingface-sentencesimilarity-bge-base-en-v1-5
huggingface-sentencesimilarity-bge-large-en
huggingface-sentencesimilarity-bge-large-en-v1-5
huggingface-sentencesimilarity-bge-large-zh-v1-5
huggingface-sentencesimilarity-bge-m3
huggingface-sentencesimilarity-bge-small-en
huggingface-sentencesimilarity-bge-small-en-v1-5
huggingface-sentencesimilarity-e5-base
huggingface-sentencesimilarity-e5-base-v2
huggingface-sentencesimilarity-e5-large
huggingface-sentencesimilarity-e5-large-v2
huggingface-sentencesimilarity-e5-small-v2
huggingface-sentencesimilarity-multilingual-e5-base
huggingface-sentencesimilarity-multilingual-e5-large
huggingface-textembedding-all-MiniLM-L12-v2
huggingface-textembedding-all-MiniLM-L6-v2
huggingface-textembedding-bge-base-en-v1-5
huggingface-textembedding-bloom-7b1
huggingface-textembedding-bloom-7b

In [16]:
session = sagemaker.Session()

role = sagemaker.get_execution_role()

region = boto3.Session().region_name

print("Region :", region)
print("Role   :", role)

Region : us-east-1
Role   : arn:aws:iam::334590195171:role/genaiapp


In [19]:
MODEL_ID = "huggingface-sentencesimilarity-bge-large-en-v1-5"
MODEL_VERSION = "1.1.21"

INSTANCE_TYPE = "ml.m5.large"

ENDPOINT_NAME = "edi-bge-embedding"

In [20]:
from sagemaker.jumpstart.model import JumpStartModel

model = JumpStartModel(model_id=MODEL_ID)

print(model)

JumpStartModel: {'model_type': <JumpStartModelType.OPEN_WEIGHTS: 'open_weights'>, '_model_data_is_set': False, 'model_id': 'huggingface-sentencesimilarity-bge-large-en-v1-5', 'model_version': '*', 'instance_type': 'ml.g5.2xlarge', 'resources': <sagemaker.compute_resource_requirements.resource_requirements.ResourceRequirements object at 0x7f0352ff96c0>, 'tolerate_vulnerable_model': False, 'tolerate_deprecated_model': False, 'region': 'us-east-1', 'sagemaker_session': <sagemaker.session.Session object at 0x7f0352ed1a50>, 'role': 'arn:aws:iam::334590195171:role/genaiapp', 'model_data': {'S3DataSource': {'S3Uri': 's3://jumpstart-cache-prod-us-east-1/huggingface-sentencesimilarity/huggingface-sentencesimilarity-bge-large-en-v1-5/artifacts/inference-prepack/v1.0.0/', 'S3DataType': 'S3Prefix', 'CompressionType': 'None'}}, 'image_uri': '763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-inference:1.13.1-transformers4.26.0-gpu-py39-cu117-ubuntu20.04', 'predictor_cls': <class 'sage

In [21]:
from sagemaker.jumpstart.model import JumpStartModel

model = JumpStartModel(
    model_id=MODEL_ID,
    model_version=MODEL_VERSION,
    role=role
)

print("✅ Model Loaded Successfully")

✅ Model Loaded Successfully


In [23]:
MODEL_ID = "huggingface-textembedding-all-MiniLM-L6-v2"

model = JumpStartModel(
    model_id=MODEL_ID,
    role=role
)

print(model)

Using model 'huggingface-textembedding-all-MiniLM-L6-v2' with wildcard version identifier '*'. You can pin to version '2.0.16' for more stable results. Note that models may have different input/output signatures after a major version upgrade.


JumpStartModel: {'model_type': <JumpStartModelType.OPEN_WEIGHTS: 'open_weights'>, '_model_data_is_set': False, 'model_id': 'huggingface-textembedding-all-MiniLM-L6-v2', 'model_version': '*', 'instance_type': 'ml.g5.xlarge', 'resources': <sagemaker.compute_resource_requirements.resource_requirements.ResourceRequirements object at 0x7f0352ebc880>, 'tolerate_vulnerable_model': False, 'tolerate_deprecated_model': False, 'region': 'us-east-1', 'sagemaker_session': <sagemaker.session.Session object at 0x7f0352ebcbb0>, 'role': 'arn:aws:iam::334590195171:role/genaiapp', 'model_data': {'S3DataSource': {'S3Uri': 's3://jumpstart-cache-prod-us-east-1/huggingface-textembedding/huggingface-textembedding-all-MiniLM-L6-v2/artifacts/inference-prepack/v2.0.0/', 'S3DataType': 'S3Prefix', 'CompressionType': 'None'}}, 'image_uri': '683313688378.dkr.ecr.us-east-1.amazonaws.com/tei:2.0.1-tei1.4.0-gpu-py310-cu122-ubuntu22.04', 'predictor_cls': <class 'sagemaker.base_predictor.Predictor'>, 'name': 'hf-textembe

In [24]:
import json
import numpy as np
import pandas as pd
from tqdm import tqdm

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [25]:
import json

with open("../data/chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

print("="*60)
print("Chunks Loaded :", len(chunks))
print("="*60)

print(chunks[0].keys())

Chunks Loaded : 292
dict_keys(['chunk_id', 'page', 'text', 'word_count'])


In [26]:
!pip install --no-cache-dir sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 201.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 366.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 765.1/765.1 kB 615.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 456.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 232.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 794.1/794.1 kB 521.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 373.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.7/37.7 MB 232.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.1/532.1 MB 142.6 MB/s  0:00:03eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 395.9 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 360.0 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 355.6 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━

In [27]:
import transformers
import torch

print(transformers.__version__)
print(torch.__version__)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 import transformers                                                                          │
│   2 import torch                                                                                 │
│   3                                                                                              │
│   4 print(transformers.__version__)                                                              │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
ModuleNotFoundError: No module named 'transformers'

In [28]:
import torch

print(torch.__version__)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 import torch                                                                                 │
│   2                                                                                              │
│   3 print(torch.__version__)                                                                     │
│   4                                                                                              │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
ModuleNotFoundError: No module named 'torch'

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

print("Embedding Model Loaded")